In [ ]:
# --- portable setup (added 2026-09-12 when the project moved to GitHub) ---------------------
# All paths below resolve from HME_ROOT, the repository root.  On Colab: mount Drive and point
# HME_ROOT at your clone.  Locally: run jupyter from the repo, or export HME_ROOT=/path/to/repo.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    HME_ROOT = os.environ.get('HME_ROOT', '/content/drive/MyDrive/hyperbolic-icd10')   # <-- edit
except ImportError:
    HME_ROOT = os.environ.get('HME_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
assert os.path.isdir(os.path.join(HME_ROOT, 'data')), f'HME_ROOT={HME_ROOT!r} is not the repo root'
print('HME_ROOT =', HME_ROOT)


# V2 Overnight — Does Position-Dependent Curvature Earn an *Unconfounded* Result?
**Built against your CURRENT architecture** (`train3`/`eval3`/`eval_closure` from `temperature_experiments_clean.ipynb`), not the June line-integral stack. Same hyperparameters, same loss, same checkpoint format, same `CKPT_DIR` — so it **reuses every checkpoint you already trained** (`msld_*`, `ms_const_*`, `temp_*`, `d5hi_const_*`).

**The reframe this notebook implements:** your graded model already beat constant on reconstruction MAP at d=5 with the temperature swept per-condition (0.8403 ± 0.0011 vs const's best 0.7992 ± 0.0038) — i.e., the scale-matched recon contest is *already won*. But your own paper establishes reconstruction MAP as the confounded metric. So the question that decides whether "position-dependent curvature works" is:

> **At each mode's own best temperature, does graded beat constant on CLOSURE (generalization to non-direct ancestors) — and does the effect require the real branching feature?**

**Phase A (≈30–60 min, run before sleeping — may answer the headline by itself):** evaluate every existing checkpoint on recon + closure. Best-s per mode is chosen on a *selection* closure set (seed 12, disjoint) and reported on the *test* closure set (seed 11 — the paper's), so temperature selection never touches the reported pairs.

**Phase B (overnight training, ~13–16 runs × 1500 epochs):**
- B1 — feature-anchoring ablation at the winning config (d=5, s=2.23, α=−2): κ ∈ {shuffled, random, binary leaf/internal} × seeds {0,1,2}. Real-κ runs already exist (`msld_graded_d5_x2.23_seed*`) — not retrained.
- B2 — graded d=10 seeds 1,2 at s ∈ {1.0, 2.23} (closure error bars; currently single-seed).
- B3 (optional) — graded d=5 at s=8 × 3 seeds (checks the win isn't a temperature-grid artifact).

**Decision table (pre-registered):**
| Outcome | Meaning |
|---|---|
| Graded > const on test-closure at per-mode best s, AND real > shuffled | genuine structural result on the honest metric — a real second contribution |
| Graded > const on closure, but real ≈ shuffled ≈ binary | "a leaf/internal partition helps generalization" — narrowed, still real |
| Graded ≤ const on closure | the bandwidth criterion confirmed on the metric that matters; the d=5 recon win becomes the paper's worked example of the confound |

Resume-safe: training skips any existing checkpoint; Phase A/C results re-save on every run.
Known vestigial issue, not fixed here: `loss_only_scale` in your original `train3` has identical branches (dead flag) — harmless because distance-scaling and logit-scaling are the same operation in this loss, but delete it from the paper code.

In [1]:
# CELL 1 — setup (verbatim conventions from temperature_experiments_clean)
!pip install geoopt -q
import os, pickle, json, time, numpy as np, torch, torch.nn as nn, geoopt
from collections import defaultdict, deque

ROOT_DIR=HME_ROOT
DATA_DIR=os.path.join(ROOT_DIR,'data/processed')
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR, exist_ok=True)
RESULTS_DIR=os.path.join(ROOT_DIR,'results','v2_closure_contest'); os.makedirs(RESULTS_DIR, exist_ok=True)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
with open(os.path.join(DATA_DIR,'icd10_tree_with_features.pkl'),'rb') as f: data=pickle.load(f)
nodes=data['nodes']; edges=data['edges']; features=data['features']
codes=list(nodes.keys()); code_to_idx={c:i for i,c in enumerate(codes)}
idx_to_code={i:c for c,i in code_to_idx.items()}
N=len(codes); edges_idx=[(code_to_idx[p],code_to_idx[c]) for p,c in edges]
ROOT=code_to_idx['ROOT']
print(f'N={N}, edges={len(edges_idx)}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 9.4 MB/s eta 0:00:00
device cuda
N=46817, edges=46816


In [2]:
# CELL 2 — the stack (your components verbatim + kappa_override, checkpoint reuse, disjoint closure sel/test)
# ============================================================
# V2 STACK — current architecture (train3 / eval3 / eval_closure),
# faithful to temperature_experiments_clean.ipynb, with three additions:
#   (1) kappa_override in train3  (enables the feature-anchoring ablation)
#   (2) skip-if-checkpoint-exists (resume safety / reuse of old runs)
#   (3) TWO closure sets: SELECTION (seed 12) and TEST (seed 11 = the paper's)
#       so choosing the best temperature never touches the reported pairs.
# ============================================================
import os, json, time
import numpy as np
import torch, torch.nn as nn
import geoopt
from collections import defaultdict, deque

def build_context(N, edges_idx, features, idx_to_code, device, ROOT,
                  eval_size=1000, dpair_srcs=2000):
    ctx = {}
    nbrs = defaultdict(set)
    for u, v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
    _rng = np.random.default_rng(42)
    EVAL = [edges_idx[i] for i in _rng.choice(len(edges_idx), size=min(eval_size, len(edges_idx)), replace=False)]
    _adj = defaultdict(list)
    for u, v in edges_idx: _adj[u].append(v); _adj[v].append(u)
    _r2 = np.random.default_rng(0); DPAIRS = []
    for s in _r2.integers(0, N, size=dpair_srcs):
        dd = {int(s): 0}; q = deque([int(s)])
        while q:
            x = q.popleft()
            for y in _adj[x]:
                if y not in dd: dd[y] = dd[x]+1; q.append(y)
        t = int(_r2.integers(0, N))
        if t in dd and t != s: DPAIRS.append((int(s), t, dd[t]))
    parent_of = {}
    for u, v in edges_idx: parent_of[v] = u
    def ancestors(x):
        out = []; cur = x
        while cur in parent_of:
            cur = parent_of[cur]; out.append(cur)
        return out
    def closure_set(seed, n_src=4000, n_pairs=1000):
        rt = np.random.default_rng(seed)
        pairs = []
        for x in rt.choice(N, size=min(n_src, N), replace=False):
            anc = ancestors(int(x))
            for a in anc[1:]:
                pairs.append((int(x), a))
        rt.shuffle(pairs)
        return pairs[:n_pairs]
    CLOSURE_TEST = closure_set(11)                 # the paper's set — reported
    _test_set = set(CLOSURE_TEST)
    CLOSURE_SEL = [p for p in closure_set(12, n_pairs=3000)
                   if p not in _test_set][:1000]   # selection-only, disjoint from test
    if len(CLOSURE_SEL) < 200:                     # tiny graphs: not enough pairs
        print(f"WARNING: only {len(CLOSURE_SEL)} disjoint selection pairs; "
              f"falling back to overlapping seed-12 set (fine for smoke tests, "
              f"NOT for reported numbers)")
        CLOSURE_SEL = closure_set(12)
    ctx.update(nbrs=nbrs, EVAL=EVAL, DPAIRS=DPAIRS,
               CLOSURE_TEST=CLOSURE_TEST, CLOSURE_SEL=CLOSURE_SEL,
               direct_nbrs=nbrs, N=N, device=device, ROOT=ROOT)
    bf_all = np.array([features[idx_to_code[i]]['branching_factor'] for i in range(N)])
    ctx['bf_all'] = bf_all
    child_count = np.zeros(N, dtype=int)
    for (p, c) in edges_idx: child_count[p] += 1
    ctx['is_leaf'] = (child_count == 0)
    ctx['edges_idx'] = edges_idx
    ctx['connected'] = set(edges_idx) | set((v, u) for u, v in edges_idx)
    return ctx

# ---------- kappa (verbatim law) + ablation variants ----------
def graded_kappa(ctx, scale=1.0):
    k = np.log1p(ctx['bf_all'])**2
    k = 2*(k-k.min())/(k.max()-k.min()+1e-9)-1
    return torch.tensor(k*scale, dtype=torch.float32, device=ctx['device'])

def kappa_variant(ctx, mode, seed=0):
    """real / shuffled (same values, permuted node assignment) /
    random (matched to real's mean+std) / binary (hard leaf/internal at the
    real kappa's two modal levels: leaves=-1, internal=+distribution mean of internal)."""
    base = graded_kappa(ctx).cpu().numpy()
    rng = np.random.default_rng(seed)
    if mode == 'real':
        k = base
    elif mode == 'shuffled':
        k = base[rng.permutation(len(base))]
    elif mode == 'random':
        k = rng.normal(base.mean(), base.std(), size=len(base))
        k = np.clip(k, -1, 1)
    elif mode == 'binary':
        internal_mean = base[~ctx['is_leaf']].mean()
        k = np.where(ctx['is_leaf'], -1.0, float(internal_mean))
    else:
        raise ValueError(mode)
    return torch.tensor(k, dtype=torch.float32, device=ctx['device'])

# ---------- eval3 (verbatim: MAP/ranks on EVAL + scale-fitted distortion) ----------
def _dpoin(a, allp):
    diff2 = np.sum((allp-a)**2, axis=1); nu = 1-np.sum(a**2); nv = 1-np.sum(allp**2, axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12), 1.0))
def _deuc(a, allp): return np.linalg.norm(allp-a, axis=1)

def eval3(ctx, pos, euclidean=False):
    if np.isnan(pos).any():
        return {'MAP': float('nan'), 'mean_rank': float('nan'),
                'median_rank': float('nan'), 'distortion': float('nan')}
    dfn = _deuc if euclidean else _dpoin
    ranks = []; aps = []
    for (u, v) in ctx['EVAL']:
        d = dfn(pos[u], pos); d[u] = np.inf; order = np.argsort(d)
        ranks.append(int(np.where(order == v)[0][0])+1)
        truth = ctx['nbrs'][u]; hit = 0; precs = []
        for j, node in enumerate(order):
            if node in truth: hit += 1; precs.append(hit/(j+1))
            if hit == len(truth): break
        if precs: aps.append(np.mean(precs))
    emb_d = np.array([float(dfn(pos[u], pos[v:v+1])[0]) for (u, v, dg) in ctx['DPAIRS']])
    grf_d = np.array([dg for (u, v, dg) in ctx['DPAIRS']], dtype=float)
    c = np.dot(emb_d, grf_d)/(np.dot(emb_d, emb_d)+1e-12)
    dist = float(np.mean(np.abs(c*emb_d-grf_d)/grf_d))
    ranks = np.array(ranks)
    return {'MAP': float(np.mean(aps)), 'mean_rank': float(np.mean(ranks)),
            'median_rank': float(np.median(ranks)), 'distortion': dist}

def eval_closure(ctx, pos, which='test', euclidean=False):
    pairs = ctx['CLOSURE_TEST'] if which == 'test' else ctx['CLOSURE_SEL']
    dfn = _deuc if euclidean else _dpoin
    rr = []; ranks = []; h10 = []
    for (x, a) in pairs:
        d = dfn(pos[x], pos); d[x] = np.inf
        for w in ctx['direct_nbrs'][x]: d[w] = np.inf
        r = int(np.where(np.argsort(d) == a)[0][0])+1
        ranks.append(r); rr.append(1.0/r); h10.append(r <= 10)
    return {'MRR': float(np.mean(rr)), 'mean_rank': float(np.mean(ranks)),
            'median_rank': float(np.median(ranks)), 'hits@10': float(np.mean(h10))}

# ---------- negatives (verbatim) ----------
def sample_negs(ctx, anchors, K):
    N = ctx['N']; connected = ctx['connected']
    out = np.random.randint(0, N, size=(len(anchors), K))
    for i, a in enumerate(anchors):
        for j in range(K):
            while out[i, j] == a or (a, out[i, j]) in connected:
                out[i, j] = np.random.randint(0, N)
    return out

# ---------- model (verbatim) ----------
class PoincareEmbedding(nn.Module):
    def __init__(self, N, dim, init_scale=1e-3):
        super().__init__()
        self.manifold = geoopt.PoincareBall(c=1.0)
        self.embeddings = geoopt.ManifoldParameter(
            torch.empty(N, dim).uniform_(-init_scale, init_scale), manifold=self.manifold)
    def forward(self, idx): return self.embeddings[idx]

# ---------- train3: faithful, + kappa_override + skip-if-exists ----------
def train3(ctx, CKPT_DIR, mode, scale=1.0, dim=10, alpha=-2.0, epochs=1500,
           lr=50, K=50, seed=0, tag=None, log_every=300, verbose=True,
           kappa_override=None, batch_size=1024, reuse_existing=True):
    """Identical hyperparameters, loss, hierarchy penalty, ROOT pin, burn-in,
    and best-epoch-on-EVAL selection as the paper runs. Additions:
      kappa_override : custom kappa tensor for mode='graded' (ablation)
      reuse_existing : if the checkpoint exists, load it instead of retraining
    NOTE: best-epoch is selected on reconstruction (EVAL) exactly as before, so
    closure remains an untouched metric for every checkpoint, old and new."""
    device = ctx['device']; N = ctx['N']; ROOT = ctx['ROOT']
    torch.manual_seed(seed); np.random.seed(seed)
    tag = tag or f'{mode}_x{scale}_d{dim}_a{alpha}_s{seed}'
    ckpt = f'{CKPT_DIR}/{tag}.pt'
    if reuse_existing and os.path.exists(ckpt):
        ck = torch.load(ckpt, map_location='cpu')
        best = ck.get('metrics', {}); best['tag'] = tag; best['reused'] = True
        if verbose: print(f"  {tag}: reused existing checkpoint")
        return best
    kf = (kappa_override if kappa_override is not None else graded_kappa(ctx)).detach()
    model = PoincareEmbedding(N, dim, 0.001).to(device)
    opt = geoopt.optim.RiemannianSGD(model.parameters(), lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT] = 0.0
    E = np.array(ctx['edges_idx']); best = {'MAP': 0}; bestpos = None
    for ep in range(epochs):
        eff = lr*0.01 if ep < 10 else lr
        for pg in opt.param_groups: pg['lr'] = eff
        perm = np.random.permutation(len(E))
        for bs in range(0, len(E), batch_size):
            b = E[perm[bs:bs+batch_size]]
            a = torch.tensor(b[:, 0]).to(device); p = torch.tensor(b[:, 1]).to(device)
            ng = torch.tensor(sample_negs(ctx, b[:, 0], K)).to(device)
            ea = model.embeddings[a]; ep_ = model.embeddings[p]; en = model.embeddings[ng]
            bp = model.manifold.dist(ea, ep_); bn = model.manifold.dist(ea.unsqueeze(1), en)
            if mode == 'const':
                dp, dn = bp, bn
            else:
                dp = bp*torch.exp(alpha*0.5*(kf[a]+kf[p]))
                dn = bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            s = float(scale)
            dp_l, dn_l = dp*s, dn*s
            nk = (dp_l+torch.logsumexp(-torch.cat([dp_l.unsqueeze(1), dn_l], 1), 1)).mean()
            an = ea.norm(dim=-1); pn = ep_.norm(dim=-1)
            loss = nk + 1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT] = 0.0
            opt.step()
        if (ep+1) % log_every == 0:
            pos = model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any():
                print(f'  {tag} NaN ep{ep+1}'); break
            r = eval3(ctx, pos)
            if r['MAP'] > best['MAP']: best = r; bestpos = pos.copy()
    torch.save({'emb': torch.tensor(bestpos), 'tag': tag, 'metrics': best}, ckpt)
    best['tag'] = tag; best['reused'] = False
    if verbose:
        print(f"  {tag}: MAP {best['MAP']:.4f} mean_rk {best['mean_rank']:.0f} "
              f"med_rk {best['median_rank']:.0f} distort {best['distortion']:.4f}")
    return best

def load_emb(CKPT_DIR, tag):
    ck = torch.load(f'{CKPT_DIR}/{tag}.pt', map_location='cpu')
    return ck['emb'].numpy(), ck.get('metrics', {})

# ---------- full report on one checkpoint (recon + closure sel/test) ----------
def full_report(ctx, CKPT_DIR, tag, euclidean=False):
    try:
        pos, _ = load_emb(CKPT_DIR, tag)
    except Exception:
        return None
    r = eval3(ctx, pos, euclidean=euclidean)
    c_sel = eval_closure(ctx, pos, 'sel', euclidean=euclidean)
    c_tst = eval_closure(ctx, pos, 'test', euclidean=euclidean)
    return {'tag': tag, 'recon': r, 'closure_sel': c_sel, 'closure_test': c_tst}


ctx = build_context(N, edges_idx, features, idx_to_code, device, ROOT)
print(f"EVAL={len(ctx['EVAL'])}  DPAIRS={len(ctx['DPAIRS'])}  "
      f"closure_test={len(ctx['CLOSURE_TEST'])}  closure_sel={len(ctx['CLOSURE_SEL'])}")
assert not (set(ctx['CLOSURE_TEST']) & set(ctx['CLOSURE_SEL'])), 'closure sets must be disjoint'
print('closure selection/test sets disjoint ✓')

EVAL=1000  DPAIRS=2000  closure_test=1000  closure_sel=1000
closure selection/test sets disjoint ✓


In [3]:
# CELL 3 — PHASE A: evaluate every existing checkpoint (no training).
# Appends to Drive as it goes; re-running skips already-evaluated tags.
PHASE_A_PATH = os.path.join(RESULTS_DIR, 'phaseA_checkpoint_reports.json')
reports = {}
if os.path.exists(PHASE_A_PATH):
    with open(PHASE_A_PATH) as f: reports = json.load(f)
    print(f'loaded {len(reports)} existing reports')

TAGS = []
for s in [1.0, 2.23, 5.0]:
    for seed in [0,1,2]:
        for dim in [5, 2]:
            TAGS.append((f'msld_const_d{dim}_x{s}_seed{seed}', False))
            TAGS.append((f'msld_graded_d{dim}_x{s}_seed{seed}', False))
for s in [1.0, 2.23, 3.0, 5.0, 8.0]:
    for seed in [0,1,2]:
        TAGS.append((f'ms_const_x{s}_seed{seed}', False))
    TAGS.append((f'temp_graded_x{s}', False))
    TAGS.append((f'temp_const_x{s}', False))
for s in [8.0, 12.0]:
    for seed in [0,1,2]:
        TAGS.append((f'd5hi_const_x{s}_seed{seed}', False))
for s in [1.0, 3.0, 8.0]:
    TAGS.append((f'temp_euclid_x{s}', True))
for s in [1.0, 0.5, 0.25, 0.1]:
    TAGS.append((f'sub_const_x{s}', False))

t0=time.time(); found=0; missing=[]
for tag, euc in TAGS:
    if tag in reports: found+=1; continue
    r = full_report(ctx, CKPT_DIR, tag, euclidean=euc)
    if r is None: missing.append(tag); continue
    reports[tag] = r; found+=1
    with open(PHASE_A_PATH,'w') as f: json.dump(reports, f, indent=1)
    print(f"  {tag:34s} reconMAP {r['recon']['MAP']:.4f}  "
          f"closMRR(sel) {r['closure_sel']['MRR']:.4f}  "
          f"closMRR(test) {r['closure_test']['MRR']:.4f}  "
          f"distort {r['recon']['distortion']:.4f}   [{(time.time()-t0)/60:.0f} min]")
print(f"\nPhase A: {found} evaluated, {len(missing)} missing checkpoints")
if missing: print('missing:', missing)


Phase A: 0 evaluated, 74 missing checkpoints
missing: ['msld_const_d5_x1.0_seed0', 'msld_graded_d5_x1.0_seed0', 'msld_const_d2_x1.0_seed0', 'msld_graded_d2_x1.0_seed0', 'msld_const_d5_x1.0_seed1', 'msld_graded_d5_x1.0_seed1', 'msld_const_d2_x1.0_seed1', 'msld_graded_d2_x1.0_seed1', 'msld_const_d5_x1.0_seed2', 'msld_graded_d5_x1.0_seed2', 'msld_const_d2_x1.0_seed2', 'msld_graded_d2_x1.0_seed2', 'msld_const_d5_x2.23_seed0', 'msld_graded_d5_x2.23_seed0', 'msld_const_d2_x2.23_seed0', 'msld_graded_d2_x2.23_seed0', 'msld_const_d5_x2.23_seed1', 'msld_graded_d5_x2.23_seed1', 'msld_const_d2_x2.23_seed1', 'msld_graded_d2_x2.23_seed1', 'msld_const_d5_x2.23_seed2', 'msld_graded_d5_x2.23_seed2', 'msld_const_d2_x2.23_seed2', 'msld_graded_d2_x2.23_seed2', 'msld_const_d5_x5.0_seed0', 'msld_graded_d5_x5.0_seed0', 'msld_const_d2_x5.0_seed0', 'msld_graded_d2_x5.0_seed0', 'msld_const_d5_x5.0_seed1', 'msld_graded_d5_x5.0_seed1', 'msld_const_d2_x5.0_seed1', 'msld_graded_d2_x5.0_seed1', 'msld_const_d5_x5.0_

In [4]:
# CELL 4 — PHASE A verdict preview: the matched-temperature closure contest.
# Per mode+dim: pick best s on closure_sel (mean over seeds), report closure_test.
def agg(tags):
    rs = [reports[t] for t in tags if t in reports]
    if not rs: return None
    def m(path):
        vals = [r[path[0]][path[1]] for r in rs]
        return float(np.mean(vals)), float(np.std(vals)), len(vals)
    return {'sel_MRR': m(('closure_sel','MRR')), 'test_MRR': m(('closure_test','MRR')),
            'test_h10': m(('closure_test','hits@10')), 'MAP': m(('recon','MAP')),
            'dist': m(('recon','distortion'))}

def contest(dim, const_fmt, graded_fmt, s_grid, seeds):
    print(f"\n===== d={dim}: per-mode best-s (chosen on SELECTION closure) =====")
    print(f"{'mode':>8}{'s*':>7}{'testMRR':>16}{'hits@10':>10}{'reconMAP':>16}{'distort':>10}")
    best = {}
    for mode, fmt in [('const', const_fmt), ('graded', graded_fmt)]:
        rows = {}
        for s in s_grid:
            a = agg([fmt.format(s=s, seed=sd) for sd in seeds])
            if a: rows[s] = a
        if not rows: print(f"{mode:>8}   (no checkpoints)"); continue
        s_star = max(rows, key=lambda s: rows[s]['sel_MRR'][0])
        a = rows[s_star]; best[mode] = (s_star, a)
        print(f"{mode:>8}{s_star:>7}{a['test_MRR'][0]:>10.4f} ±{a['test_MRR'][1]:.4f}"
              f"{a['test_h10'][0]:>10.3f}{a['MAP'][0]:>10.4f} ±{a['MAP'][1]:.4f}"
              f"{a['dist'][0]:>10.4f}")
        for s in sorted(rows):
            r = rows[s]
            print(f"          s={s:<5} selMRR {r['sel_MRR'][0]:.4f}  "
                  f"testMRR {r['test_MRR'][0]:.4f}  MAP {r['MAP'][0]:.4f}  (n={r['MAP'][2]})")
    if 'const' in best and 'graded' in best:
        d = best['graded'][1]['test_MRR'][0] - best['const'][1]['test_MRR'][0]
        pooled = (best['graded'][1]['test_MRR'][1] + best['const'][1]['test_MRR'][1]) / 2
        print(f"  → graded − const on TEST closure MRR: {d:+.4f}  (pooled seed-std ≈ {pooled:.4f})")
    return best

best_d5  = contest(5,  'msld_const_d5_x{s}_seed{seed}',  'msld_graded_d5_x{s}_seed{seed}',  [1.0,2.23,5.0], [0,1,2])
best_d2  = contest(2,  'msld_const_d2_x{s}_seed{seed}',  'msld_graded_d2_x{s}_seed{seed}',  [1.0,2.23,5.0], [0,1,2])
# d=10: const is multi-seed, graded currently single-seed (B2 adds seeds tonight)
print("\n===== d=10 (graded single-seed until Phase B2 finishes) =====")
for s in [1.0,2.23,3.0,5.0,8.0]:
    a_c = agg([f'ms_const_x{s}_seed{sd}' for sd in [0,1,2]])
    a_g = agg([f'temp_graded_x{s}'])
    cc = f"const testMRR {a_c['test_MRR'][0]:.4f}±{a_c['test_MRR'][1]:.4f}" if a_c else "const —"
    gg = f"graded testMRR {a_g['test_MRR'][0]:.4f} (n=1)" if a_g else "graded —"
    print(f"  s={s:<5} {cc}   {gg}")


===== d=5: per-mode best-s (chosen on SELECTION closure) =====
    mode     s*         testMRR   hits@10        reconMAP   distort
   const   (no checkpoints)
  graded   (no checkpoints)

===== d=2: per-mode best-s (chosen on SELECTION closure) =====
    mode     s*         testMRR   hits@10        reconMAP   distort
   const   (no checkpoints)
  graded   (no checkpoints)

===== d=10 (graded single-seed until Phase B2 finishes) =====
  s=1.0   const —   graded —
  s=2.23  const —   graded —
  s=3.0   const —   graded —
  s=5.0   const —   graded —
  s=8.0   const —   graded —


In [ ]:
# CELL 5 — PHASE B: overnight training queue (resume-safe; skips existing checkpoints).
EPOCHS = 1500          # your standard; lower to 1000 if the night is short
QUEUE = []
# B1 — feature-anchoring ablation at the winning config (d=5, s=2.23, alpha=-2)
for mode_k in ['shuffled', 'random', 'binary']:
    for seed in [0,1,2]:
        QUEUE.append(dict(kind='ablate', kmode=mode_k, dim=5, s=2.23, seed=seed,
                          tag=f'v2ab_{mode_k}_d5_x2.23_seed{seed}'))
# B2 — graded d=10 multi-seed (closure error bars)
for s in [1.0, 2.23]:
    for seed in [1,2]:
        QUEUE.append(dict(kind='graded', kmode='real', dim=10, s=s, seed=seed,
                          tag=f'v2g_d10_x{s}_seed{seed}'))
# B3 — optional: graded d=5 at const's high-s region
for seed in [0,1,2]:
    QUEUE.append(dict(kind='graded', kmode='real', dim=5, s=8.0, seed=seed,
                      tag=f'v2g_d5_x8.0_seed{seed}'))

print(f"{len(QUEUE)} runs queued (skips any with existing checkpoints)")
t0=time.time()
for i,q in enumerate(QUEUE):
    print(f"\n===== run {i+1}/{len(QUEUE)}: {q['tag']}  (elapsed {(time.time()-t0)/3600:.1f} h) =====")
    try:
        ko = None if q['kmode']=='real' else kappa_variant(ctx, q['kmode'], seed=q['seed'])
        r = train3(ctx, CKPT_DIR, 'graded', scale=q['s'], dim=q['dim'], alpha=-2.0,
                   epochs=EPOCHS, seed=q['seed'], tag=q['tag'], kappa_override=ko)
        rep = full_report(ctx, CKPT_DIR, q['tag'])
        with open(PHASE_A_PATH) as f: reports = json.load(f)
        reports[q['tag']] = rep
        with open(PHASE_A_PATH,'w') as f: json.dump(reports, f, indent=1)
        print(f"  closMRR(test) {rep['closure_test']['MRR']:.4f}  "
              f"reconMAP {rep['recon']['MAP']:.4f}")
    except Exception as e:
        print(f"!! {q['tag']} failed: {e} — continuing")
print(f"\nDONE. total {(time.time()-t0)/3600:.1f} h")

16 runs queued (skips any with existing checkpoints)

===== run 1/16: v2ab_shuffled_d5_x2.23_seed0  (elapsed 0.0 h) =====
  v2ab_shuffled_d5_x2.23_seed0: MAP 0.4880 mean_rk 1773 med_rk 6 distort 0.1507
!! v2ab_shuffled_d5_x2.23_seed0 failed: [Errno 2] No such file or directory: '/content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/results/V2_closure_contest/phaseA_checkpoint_reports.json' — continuing

===== run 2/16: v2ab_shuffled_d5_x2.23_seed1  (elapsed 0.6 h) =====
  v2ab_shuffled_d5_x2.23_seed1: MAP 0.4858 mean_rk 1337 med_rk 6 distort 0.1531
!! v2ab_shuffled_d5_x2.23_seed1 failed: [Errno 2] No such file or directory: '/content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/results/V2_closure_contest/phaseA_checkpoint_reports.json' — continuing

===== run 3/16: v2ab_shuffled_d5_x2.23_seed2  (elapsed 1.2 h) =====
  v2ab_shuffled_d5_x2.

In [ ]:
# CELL 6 — PHASE C: final summary + pre-registered verdict. Rerun any time.
with open(PHASE_A_PATH) as f: reports = json.load(f)

def stat(tags, path):
    v=[reports[t][path[0]][path[1]] for t in tags if t in reports]
    return (float(np.mean(v)), float(np.std(v)), len(v)) if v else None

print("=== ABLATION at d=5, s=2.23, α=−2 (the winning config) ===")
print(f"{'kappa':>10}{'reconMAP':>18}{'closMRR(test)':>18}{'hits@10':>10}")
abl = {}
for mode_k, tags in [('real', [f'msld_graded_d5_x2.23_seed{s}' for s in [0,1,2]]),
                     ('shuffled', [f'v2ab_shuffled_d5_x2.23_seed{s}' for s in [0,1,2]]),
                     ('random', [f'v2ab_random_d5_x2.23_seed{s}' for s in [0,1,2]]),
                     ('binary', [f'v2ab_binary_d5_x2.23_seed{s}' for s in [0,1,2]]),
                     ('const', [f'msld_const_d5_x2.23_seed{s}' for s in [0,1,2]])]:
    m = stat(tags, ('recon','MAP')); c = stat(tags, ('closure_test','MRR'))
    h = stat(tags, ('closure_test','hits@10'))
    if m:
        abl[mode_k] = (m, c)
        print(f"{mode_k:>10}{m[0]:>11.4f} ±{m[1]:.4f}{c[0]:>12.4f} ±{c[1]:.4f}{h[0]:>10.3f}")
    else:
        print(f"{mode_k:>10}   (pending)")

print("\n=== d=10 graded multi-seed closure (B2) ===")
for s in [1.0, 2.23]:
    tags = [f'temp_graded_x{s}'] + [f'v2g_d10_x{s}_seed{sd}' for sd in [1,2]]
    c = stat(tags, ('closure_test','MRR')); m = stat(tags, ('recon','MAP'))
    cc = stat([f'ms_const_x{s}_seed{sd}' for sd in [0,1,2]], ('closure_test','MRR'))
    if c and cc:
        print(f"  s={s}: graded closMRR {c[0]:.4f}±{c[1]:.4f} (n={c[2]})  "
              f"vs const {cc[0]:.4f}±{cc[1]:.4f}")

print("\n----- VERDICT (pre-registered) -----")
if 'real' in abl and 'const' in abl:
    g_c = abl['real'][1]; c_c = abl['const'][1]
    gap = g_c[0]-c_c[0]; noise = (g_c[1]+c_c[1])/2
    if gap > 2*noise:
        line1 = "Graded beats const on TEST closure at the matched config → real generalization effect."
        if 'shuffled' in abl and abl['real'][1][0]-abl['shuffled'][1][0] > 2*noise:
            print(line1); print("AND real features beat shuffled → the branching anchoring is doing the work.")
            print("→ Position-dependent curvature earns a genuine, unconfounded second contribution.")
        elif 'shuffled' in abl:
            print(line1); print("BUT real ≈ shuffled/binary → narrowed claim: a coarse partition, not the branching law.")
        else:
            print(line1 + " (ablation pending)")
    else:
        print(f"Graded − const on TEST closure = {gap:+.4f} vs seed noise ~{noise:.4f} → no generalization win.")
        print("The bandwidth criterion is confirmed on the metric that matters.")
        print("The d=5 recon win (0.8403 vs 0.7992) goes in the paper as the worked example")
        print("of a method gain that exists only on the confounded metric.")
else:
    print("(run Phases A and B first)")

## After the run
1. Paste Cells 4 and 6 output back. Whichever verdict row fires, the corresponding paper text is pre-committed above — no post-hoc framing.
2. Numbers destined for a table still need: recon MAP is best-epoch-selected on EVAL (blocker B1 applies to *recon* numbers; closure numbers are untouched by selection and are clean). Distortion here is the scale-fitted sampled version — label it (B7).
3. If graded wins closure: the immediate next experiment is κ-GCN under matched scale (the reviewer-named gap) — a win there and you have a full method paper section.
4. If graded loses closure: E-queue continues as planned (κ-GCN, matched-optimizer Euclidean, bandwidth criterion on a high-margin-spread graph — the one place your theory predicts adaptive curvature *should* win).